# Galaxy10 DECaLS — CNN + VAE (CRISP-DM)

**Module:** H9DLGA Deep Learning and Generative AI (Repeat)

**Research question:** Given class imbalance in Galaxy10 DECaLS, do VAE-generated synthetic images improve CNN morphological classification on minority classes without degrading majority-class performance?

This notebook mirrors `python -m src.run_pipeline` and follows CRISP-DM.


## 0. Setup


In [ ]:
import sys
from pathlib import Path

# Project root = parent of notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import torch
from src import config
from src.data import (
    download_galaxy10,
    ensure_dirs,
    load_galaxy10_arrays,
    stratified_split,
    class_counts,
    make_dataloaders,
    get_cnn_transforms,
)
from src.train_vae import train_vae
from src.train_cnn import train_cnn
from src.generate import build_balanced_training_set, make_loader_from_arrays
from src.evaluate import evaluate_classifier, compare_minority_performance

ensure_dirs()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("project root:", PROJECT_ROOT)


## 1. Business Understanding

We study whether **generative augmentation (VAE)** helps a **discriminative CNN (ResNet50)** under morphological class imbalance in Galaxy10 DECaLS, with special attention to the minority class *Cigar Shaped Smooth* (class 4).


## 2. Data Understanding — download & class counts


In [ ]:
# First run downloads Galaxy10_DECals.h5 (~2.5GB) into data/raw/
h5_path = download_galaxy10()
images, labels = load_galaxy10_arrays(h5_path)
train_idx, val_idx, test_idx = stratified_split(labels)
print("Train counts:", class_counts(labels, train_idx))
print("Val counts:  ", class_counts(labels, val_idx))
print("Test counts: ", class_counts(labels, test_idx))


## 3. Data Preparation — dataloaders (VAE @64px, CNN @224px + ImageNet norm)


In [ ]:
vae_train, vae_val, _ = make_dataloaders(
    images, labels, train_idx, val_idx, test_idx, mode="vae"
)
cnn_train, cnn_val, cnn_test = make_dataloaders(
    images, labels, train_idx, val_idx, test_idx, mode="cnn"
)
print("VAE batches:", len(vae_train), "| CNN batches:", len(cnn_train))


## 4. Modeling — ConvVAE (Generative AI)

VAE formulation matches the module labs: encode → μ, logσ² → reparameterize → decode; loss = reconstruction + KL.


In [ ]:
# Reduce epochs for a quick trial; use config defaults for the full experiment.
vae = train_vae(
    vae_train,
    vae_val,
    epochs=config.VAE_EPOCHS,
    device=device,
    save_path=config.OUTPUT_MODELS / "vae_galaxy10.pt",
)


## 5. Modeling — generate synthetic images to balance classes


In [ ]:
aug_images, aug_labels, added = build_balanced_training_set(
    images, labels, train_idx, vae, device
)
print("Synthetics added:", added)
cnn_train_aug = make_loader_from_arrays(
    aug_images,
    aug_labels,
    transform=get_cnn_transforms(train=True),
    batch_size=config.CNN_BATCH_SIZE,
    shuffle=True,
)


## 6. Modeling — CNN A (real only) vs CNN B (real + VAE synthetics)

ResNet50 transfer learning follows Lab 2: train FC head, then fine-tune `layer4` + FC.


In [ ]:
cnn_real = train_cnn(
    cnn_train,
    cnn_val,
    save_path=config.OUTPUT_MODELS / "cnn_real_only.pt",
    device=device,
)
cnn_aug = train_cnn(
    cnn_train_aug,
    cnn_val,
    save_path=config.OUTPUT_MODELS / "cnn_real_plus_vae.pt",
    device=device,
)


## 7. Evaluation — held-out test comparison (answers the RQ)


In [ ]:
metrics_real = evaluate_classifier(cnn_real, cnn_test, device, tag="real_only")
metrics_aug = evaluate_classifier(cnn_aug, cnn_test, device, tag="real_plus_vae")
compare_minority_performance(metrics_real, metrics_aug)


## 8. Next deliverables

- Write IEEE paper under `paper/` using these metrics/figures
- Prepare ~7 presentation slides under `presentation/`
- Complete Q2 review of *LLM Collaboration with Multi-Agent Reinforcement Learning*
